# load_defect_log

**Purpose:** Load the project defect log into a table for reporting. The single source is
`config/defect_log.yaml` (DEC-024); `docs/defect_log.md` is built from the same file by `scripts/build_defect_log.py`.

**Input:** `Files/config/defect_log.yaml` (upload with `fab cp` after the file changes).

**Output tables:** `defect_log` (overwrite). The first six columns (`defect_id`, `found_date`, `area`, `severity`,
`status`, `description`) are the ones the semantic model and the GIS `qa_defect_log.csv` use; the others add detail.

**Run order:** independent of the pipeline (01_bronze_transform -> 02_silver_transform -> 03_gold_transform); run after the yaml changes.

In [ ]:
import yaml
from pyspark.sql import functions as F

BASE = "/lakehouse/default/Files"
with open(f"{BASE}/config/defect_log.yaml") as fh:
    CONFIG = yaml.safe_load(fh)

defects = CONFIG["defects"]
accepted = CONFIG["accepted_values"]
status_map = CONFIG["status_map"]

# Data quality checks on the config: stop before writing if any fails
problems = []
ids = [d.get("id") for d in defects]
if any(i is None for i in ids) or len(ids) != len(set(ids)):
    problems.append(f"id must be present and unique: {ids}")
for d in defects:
    for field in ["issue", "impact", "handling", "status", "area", "severity"]:
        if not d.get(field):
            problems.append(f"{d.get('id')}: {field} is empty")
    for field in ["status", "area", "severity"]:
        if d.get(field) not in accepted[field]:
            problems.append(f"{d.get('id')}: {field} = {d.get(field)!r} not in {accepted[field]}")
    if d.get("status") not in status_map:
        problems.append(f"{d.get('id')}: status {d.get('status')!r} has no entry in status_map")
if problems:
    raise ValueError("defect_log.yaml failed its checks: " + "; ".join(problems))


def key_metrics(d):
    return "; ".join(f"{m['name']} = {m['value']:,}{m.get('unit', '')} ({m['scope']})" for m in d.get("metrics", []))


rows = [(int(d["id"].split("-")[1]), d.get("found_date"), d["area"], d["severity"], status_map[d["status"]],
         d["issue"], d["id"], d["impact"], d["handling"], d["status"], key_metrics(d))
        for d in defects]

# The first six columns are used by the semantic model and by gis/qa.py; keep their names and types
schema = ("defect_id int, found_date date, area string, severity string, status string, description string, "
          "defect_code string, impact string, handling string, resolution string, key_metrics string")

(spark.createDataFrame(rows, schema)
 .write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("defect_log"))

# Reconciliation: the table holds exactly the defects in the yaml
t = spark.table("defect_log")
table_ids = {r[0] for r in t.select("defect_code").collect()}
if t.count() != len(defects) or table_ids != set(ids):
    raise ValueError(f"defect_log has {t.count()} rows {sorted(table_ids)}, yaml has {len(defects)} {sorted(ids)}")
print("defect_log rows:", t.count(), " yaml defects:", len(defects))
display(t.orderBy("defect_id"))